# Chapter 1 — Machine Learning for Trading: From Idea to Execution

The opening chapter of MLAT is conceptual. Jansen lays out the **ML4T workflow** —
the pipeline that every later chapter slots into — and frames why ML moved from
"fringe technique" to the bread-and-butter of quant trading.

We don't fit a model here. We do four things:

1. **Verify the lake** matches what we expect.
2. **Get a real feel for our universe.** Coinbase has listed 850 symbols across 7 quote currencies over 11 years. The number of *available* symbols grew from 4 in 2015 to ~830 today. We default to looking at *everything* — universe construction is a research choice we'll make per-chapter, not a default handed to us.
3. **Get oriented with BTC**, the spine of the crypto market — what does the long-run price and return distribution actually look like?
4. **Map the ML4T loop to our track.** A picture of which later chapter handles which stage of the workflow.

## Setup

In [2]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ml4t_crypto import (
    LAKE_PATH,
    connect,
    list_tables,
    load_bars,
    perf_stats,
    table_range,
    # Full universe (default) helpers
    available_symbols_on,
    quote_currency_of,
    split_by_quote,
    symbol_listing_dates,
    symbols_with_history,
    # Curated reference universe (not default — just for comparison)
    load_curated_membership,
    curated_symbols_on,
)
from ml4t_crypto.config import DEFAULT_TABLES, QUOTE_CURRENCIES

sns.set_theme(style="whitegrid", context="notebook")
pd.options.display.float_format = "{:,.4f}".format

print(f"Lake:   {LAKE_PATH}")
print(f"Exists: {LAKE_PATH.exists()}  ({LAKE_PATH.stat().st_size / 1e9:.1f} GB)")

Lake:   /Users/russellfloyd/Dropbox/NRT/nrt_dev/data/coinbase_crypto_ohlcv_lake.duckdb
Exists: True  (45.4 GB)


## 1. Sanity-check the lake

We expect OHLCV at four granularities. Let's confirm coverage and size before we
lean on any of it.

In [ ]:
tables = list_tables()
print(f"{len(tables)} tables in the lake:")
for t in tables:
    print(f"  - {t}")

In [ ]:
# Default raw OHLCV granularities (these are the *_clean variants, the full universe).
gran_summary = []
for g, t in DEFAULT_TABLES.items():
    info = table_range(t).iloc[0].to_dict()
    info["granularity"] = g
    info["table"] = t
    gran_summary.append(info)

gran_df = pd.DataFrame(gran_summary)[["granularity", "table", "first_date", "last_date", "symbols", "rows"]]
gran_df

**Reading this.** The lake holds 850 distinct symbols and 597 million 1-minute
bars across 11 years. The 1-hour table goes back further than most public crypto
datasets — useful when we get to ARCH/GARCH (Ch 9) or any model that wants long
histories. We'll mostly work with daily (1,025,743 bars) and hourly (23,553,218
bars) in the early chapters and pull from 1-minute only when we need execution-grade
resolution.

## 2. The full universe over time

The 850 symbols include everything Coinbase has ever listed — current and
delisted, every quote currency. But the *available* universe on any given day
is much smaller and grew dramatically over time. Let's see the shape.

In [ ]:
# Quote-currency breakdown of the symbol space.
with connect() as con:
    quote_breakdown = con.execute("""
        SELECT
          CASE
            WHEN symbol LIKE '%-USDC' THEN 'USDC'
            WHEN symbol LIKE '%-USD'  THEN 'USD'
            WHEN symbol LIKE '%-USDT' THEN 'USDT'
            WHEN symbol LIKE '%-EUR'  THEN 'EUR'
            WHEN symbol LIKE '%-GBP'  THEN 'GBP'
            WHEN symbol LIKE '%-BTC'  THEN 'BTC'
            WHEN symbol LIKE '%-ETH'  THEN 'ETH'
            ELSE 'other'
          END AS quote,
          COUNT(DISTINCT symbol) AS n_pairs
        FROM bars_1d_clean
        GROUP BY quote
        ORDER BY n_pairs DESC
    """).fetchdf()
quote_breakdown

**Read this carefully.** USDC and USD are roughly tied (~367 vs ~366 pairs) — which
is the Coinbase migration story: USD pairs (legacy Coinbase Pro) were largely
deprecated in favor of USDC pairs (current Coinbase Advanced). Most tokens have
both a `*-USD` listing (historical) and a `*-USDC` listing (current); for long
backtests we'll need to stitch them together. The 35 EUR, 27 USDT, 26 GBP, 24
BTC-quoted, and 5 ETH-quoted pairs are smaller niches we can include or exclude
depending on the chapter.

In [ ]:
# Number of distinct symbols active per day, full lake, by year.
with connect() as con:
    yearly = con.execute("""
        SELECT
          DATE_TRUNC('year', ts)::DATE AS year,
          COUNT(DISTINCT symbol) AS symbols_seen_in_year,
          AVG(syms_today) AS avg_symbols_per_day,
          MAX(syms_today) AS max_symbols_per_day
        FROM (
          SELECT ts, symbol,
                 COUNT(*) OVER (PARTITION BY ts) AS syms_today
          FROM bars_1d_clean
        )
        GROUP BY year ORDER BY year
    """).fetchdf()
yearly.round(1)

In [ ]:
# Symbols active per day, plotted, by quote-currency family.
with connect() as con:
    daily_by_quote = con.execute("""
        SELECT
          ts::DATE AS date,
          CASE
            WHEN symbol LIKE '%-USDC' THEN 'USDC'
            WHEN symbol LIKE '%-USD'  THEN 'USD'
            WHEN symbol LIKE '%-USDT' THEN 'USDT'
            WHEN symbol LIKE '%-EUR'  THEN 'EUR'
            WHEN symbol LIKE '%-GBP'  THEN 'GBP'
            WHEN symbol LIKE '%-BTC'  THEN 'BTC'
            WHEN symbol LIKE '%-ETH'  THEN 'ETH'
            ELSE 'other'
          END AS quote,
          COUNT(DISTINCT symbol) AS n_symbols
        FROM bars_1d_clean
        GROUP BY date, quote
    """).fetchdf()

wide = daily_by_quote.pivot_table(index="date", columns="quote", values="n_symbols", fill_value=0)
wide.index = pd.to_datetime(wide.index)

# Stack the major quotes for a clear visual.
ordered_quotes = [q for q in ["USDC", "USD", "USDT", "EUR", "GBP", "BTC", "ETH"] if q in wide.columns]
wide = wide[ordered_quotes]

fig, ax = plt.subplots(figsize=(11, 4.5))
wide.plot.area(ax=ax, alpha=0.85, linewidth=0)
ax.set_title("Symbols with daily bars, by quote currency")
ax.set_ylabel("Distinct symbols")
ax.set_xlabel("Date")
ax.legend(title="Quote", loc="upper left", ncol=2)
plt.tight_layout()
plt.show()

**The shape of the lake.** From a 2015 single-pair start (BTC-USD only), Coinbase
expanded steadily through the 2017 ICO boom (~14 symbols/day), the 2020-21
bull market (~210), the 2022 bear (still adding, ~395 average), and through
2024-26 added several hundred more. The big visual feature is the USDC wedge
appearing around 2022 and growing rapidly, while USD plateaus and starts
shrinking. That's the migration in real time.

**Practical implications for research:**

- A backtest starting in 2015 that holds "the universe" is a 1-asset strategy until 2016. Most chapters will use a start date that gives a richer cross-section — typically 2020 or 2021.
- For long-history single-asset work (vol modeling, GARCH), BTC-USD goes back to 2015. ETH-USD starts in 2016. The rest don't have meaningful history until 2018+.
- For multi-asset cross-sectional work (factor models, ML), 2021 onwards is when the universe is rich enough that cross-sectional ranks mean something.

In [ ]:
# When did each symbol first appear in the lake? (the listing-date distribution)
listings = symbol_listing_dates(granularity="1d")
listings["quote"] = listings["symbol"].apply(quote_currency_of)
print(f"{len(listings)} symbols in the daily lake.")
print()

# How many tokens were listed each year, by quote family?
listings["first_year"] = pd.to_datetime(listings["first_date"]).dt.year
listings_by_year = listings.groupby(["first_year", "quote"]).size().unstack(fill_value=0)
listings_by_year[ordered_quotes] if all(q in listings_by_year.columns for q in ordered_quotes) else listings_by_year

In [ ]:
# How long have symbols been around?
# Histogram of n_bars (one bar = one day of history).
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(listings["n_bars"], bins=60, color="C0", edgecolor="white")
ax.set_title("Distribution of per-symbol history length (daily bars)")
ax.set_xlabel("Days of data")
ax.set_ylabel("Number of symbols")
for cutoff, label in [(90, "3mo"), (365, "1yr"), (1000, "~3yr"), (2000, "~5.5yr")]:
    ax.axvline(cutoff, color="grey", ls=":", lw=1)
    ax.text(cutoff, ax.get_ylim()[1] * 0.9, label, rotation=90, va="top", ha="right", fontsize=9, color="grey")
plt.tight_layout()
plt.show()

print("Symbols by history length:")
for cutoff in [30, 90, 180, 365, 1000, 2000]:
    n = (listings["n_bars"] >= cutoff).sum()
    print(f"  ≥ {cutoff:4d} days : {n} symbols")

**Most symbols are young.** A *lot* of the 850 names in the lake have under a year
of history — they were listed recently and we don't yet have meaningful return
samples for them. In Ch 6 (cross-validation) and beyond, the minimum-history
filter we choose will significantly affect the cross-section. Standard choices:

- **≥ 90 days** — generous; includes recent listings. Useful for cross-sectional momentum where short histories can still rank.
- **≥ 365 days** — must have a full year. Standard for most factor work.
- **≥ 1000 days** — must have ~3 years. Conservative; used when fitting per-asset models (vol/GARCH/regime) that need within-asset history.

### Sanity-check against the curated reference universe

Just to anchor against the institutional-liquidity baseline — how does our full
universe relate to the curated `top50_adv10m` (top-50 ∩ ADV20 ≥ \$10M) one?

In [ ]:
today_full     = available_symbols_on("2026-05-15", quotes=None)
today_usd      = available_symbols_on("2026-05-15", quotes={"USD", "USDC"})
today_curated  = curated_symbols_on("2026-05-15")
print(f"On 2026-05-15:")
print(f"  Full universe                    : {len(today_full)} symbols")
print(f"  USD + USDC quoted only           : {len(today_usd)} symbols")
print(f"  Curated (top-50 ∩ \$10M ADV20)    : {len(today_curated)} symbols")
print()
print(f"Curated names: {today_curated}")

So today's full universe is **~840 symbols**, the USD+USDC dollar-tradeable
subset is **~730**, and the institutional-liquidity tip is **10**. We default
to the full universe; we filter explicitly when we have a reason.

## 3. The spine — BTC since 2017

BTC is the canonical reference. Every multi-asset analysis we do later will
either include it explicitly or be benchmarked against it. So: what does it
look like?

In [ ]:
btc = load_bars("1d", symbols=["BTC-USD"], start="2017-03-11").xs("BTC-USD")
btc.head()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})

ax1.semilogy(btc.index, btc["close"], color="C1", lw=1.2)
ax1.set_title("BTC-USD daily close, log scale")
ax1.set_ylabel("USD (log)")
ax1.grid(True, which="both", alpha=0.3)

rets = btc["close"].pct_change()
realized_vol = rets.rolling(30).std() * np.sqrt(365)
ax2.plot(realized_vol.index, realized_vol, color="C3", lw=1)
ax2.set_title("30-day realized vol (annualized)")
ax2.set_ylabel("σ")
ax2.set_ylim(0, 2.0)

plt.tight_layout()
plt.show()

In [ ]:
# Perf summary across a few standard windows.
windows = {
    "full":     btc["close"].pct_change().dropna(),
    "2017-19":  btc.loc["2017":"2019", "close"].pct_change().dropna(),
    "2020-21":  btc.loc["2020":"2021", "close"].pct_change().dropna(),
    "2022":     btc.loc["2022", "close"].pct_change().dropna(),
    "2023-now": btc.loc["2023":, "close"].pct_change().dropna(),
}
summary = pd.DataFrame({k: perf_stats(v, periods_per_year=365) for k, v in windows.items()}).T
summary

**What we see.**

- **Vol clusters spectacularly.** Spikes to >150% annualized in 2017 and 2020; recently more like 40-60%. The GARCH chapter (Ch 9) is going to have a *lot* to chew on here.
- **Sharpe is highly regime-dependent.** The bull years (2020-21) post Sharpes that would make any equity quant jealous; 2022 hands them back.
- **Max drawdown over the full sample is around -77%** (the 2022 unwind). Calmar tells you returns *per unit of historical worst-case loss*, which is the right number to compare against equity strategies.

## 4. The cross-section — what does dispersion across the full universe look like?

This is the *real* point of using the full universe. Cross-sectional dispersion
in returns and risk is where most ML alpha lives. Let's get a feel for it across
**every symbol with ≥1 year of data**, USD/USDC-quoted, since 2022 (a window
that excludes the 2020-21 bull but includes both the 2022 unwind and recovery).

In [ ]:
# Eligible: ≥365 days of history, USD or USDC quoted.
eligible = symbols_with_history(min_days=365, granularity="1d", quotes={"USD", "USDC"})
print(f"{len(eligible)} symbols with ≥1 year of daily history and USD/USDC quote.")
eligible.head(15)

In [ ]:
# Pull bars for the full eligible set since 2022, compute perf.
syms = eligible["symbol"].tolist()
print(f"Loading bars for {len(syms)} symbols since 2022 …")
bars_all = load_bars("1d", symbols=syms, start="2022-01-01")
print(f"  loaded {len(bars_all):,} bars")

records = []
for sym in syms:
    try:
        s = bars_all.xs(sym)
    except KeyError:
        continue
    r = s["close"].pct_change().dropna()
    if len(r) < 250:  # at least ~10 months of in-window returns
        continue
    stats = perf_stats(r, periods_per_year=365)
    stats["symbol"] = sym
    stats["n_days"] = len(r)
    records.append(stats)

cs = (
    pd.DataFrame(records)
    .set_index("symbol")
    [["n_days", "total_return", "ann_return", "ann_vol", "sharpe", "sortino", "max_drawdown"]]
)
print(f"\n{len(cs)} symbols passed the in-window filter (≥250 daily returns since 2022).")
cs.describe(percentiles=[.05, .25, .5, .75, .95]).round(3)

In [ ]:
# Dispersion charts: Sharpe, ann return, ann vol, max drawdown.
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for ax, col, color in zip(
    axes.flat,
    ["sharpe", "ann_return", "ann_vol", "max_drawdown"],
    ["C0", "C2", "C4", "C3"],
):
    data = cs[col].dropna()
    ax.hist(data, bins=50, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(data.median(), color="black", ls="--", lw=1, label=f"median = {data.median():.2f}")
    ax.set_title(f"{col} across the full eligible universe")
    ax.set_xlabel(col)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 and bottom 15 by Sharpe, just to feel the range.
top = cs.sort_values("sharpe", ascending=False).head(15)
bot = cs.sort_values("sharpe", ascending=True).head(15)

print("=== TOP 15 BY SHARPE (since 2022) ===")
print(top.to_string())
print()
print("=== BOTTOM 15 BY SHARPE (since 2022) ===")
print(bot.to_string())

**This is the point of going broad.** The cross-section is wide: Sharpes
range from clearly negative to well over 1. The dispersion is the alpha — and
it's much richer than what you see if you restrict yourself to the 10-50 most
liquid names. The trade-off, of course, is that many of these names have
brutal microstructure: large spreads, thin order books, and you may not be
able to execute the implied positions at scale. Chapter 5 (strategy evaluation)
is where we'll confront that trade-off honestly with a cost model.

**A note on survivorship.** Because we're filtering by `min_days >= 365` looking
at *all* current and historical symbols (including delisted), we're not selecting
on "is still listed today." That's the right behavior. If a symbol existed for
2 years and then got delisted, it shows up here. The flip side: when we
*backtest*, we still need to make sure the universe filter is point-in-time, not
"today's eligibility applied retroactively." Chs 4-6 cover this carefully.

## 5. The ML4T workflow, mapped to our track

Jansen's framing of the ML4T loop has six stages. Here they are with the chapter
that does the heavy lifting on each, plus what each stage looks like in our
crypto setup.

```
   ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
   │   Data       │ →  │   Features   │ →  │   Model      │
   │ acquisition  │    │ engineering  │    │   fitting    │
   └──────────────┘    └──────────────┘    └──────────────┘
                                                  ↓
   ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
   │   Live       │ ←  │   Backtest   │ ←  │  Strategy    │
   │  trading     │    │  evaluation  │    │   design     │
   └──────────────┘    └──────────────┘    └──────────────┘
```

| Stage | Crypto-specific concern | Primary chapter |
|---|---|---|
| Data acquisition | Lake already built — full Coinbase OHLCV (850 symbols, 7 quote currencies) | Ch 2 |
| Universe construction | Min-history, quote-currency, liquidity filters — chosen per chapter | Ch 4 onward |
| Feature engineering | Momentum, vol, microstructure, regime indicators (no fundamentals!) | Ch 4 |
| Model fitting | Ridge/Lasso → trees/GBM → RNN → RL agent | Chs 7, 11, 19, 22 |
| Strategy design | Position sizing, vol-targeting, portfolio construction | Chs 5, 8 |
| Backtest evaluation | Cost model (~30 bps/side baseline, higher for thin names), purged CV, MDD/Calmar | Chs 5, 6 |
| Live trading | Out of scope — we stop at paper-traded backtest results | — |

**What's different from the book's setup.** The book is built around US equities
with SEC filings, earnings transcripts, and a single asset class with regular hours.
We have:

- 24/7 markets (no overnight gaps, no exchange close)
- No fundamentals, no earnings, no analyst estimates (so chapters 14-18 NLP work doesn't translate)
- Much shorter histories (~11 years instead of ~30) and many symbols with <1 year
- Tradeable hourly granularity (the book uses daily almost exclusively until Ch 22)
- A wildly time-varying universe (4 symbols/day in 2015, 830 today) — the book quietly assumes ~500 always-tradeable names
- Listings/delistings, multiple quote currencies, and a USD→USDC migration to stitch through
- Liquidity that spans 4+ orders of magnitude across symbols on the same day

The flip side: cleaner data, no corporate actions to handle, and the lake gives
us all the raw material to build whatever universe each chapter calls for.

## 6. What's next

**Chapter 2 — Market and fundamental data.** We dive into the lake properly:
how the bars were built, what "clean" means, what the hourly vs daily trade-offs
are, USD↔USDC stitching, and how to write the universe-filtering boilerplate
that every later chapter will reuse.

Things we *consciously deferred* in Ch 1 that we'll come back to:

- **Cost model.** We used `perf_stats` on raw returns above — no slippage, no fees. That's fine for orientation but lethal for strategy evaluation. Ch 5 introduces the cost model properly, with crypto-specific spread/impact assumptions that vary by symbol liquidity.
- **Train/test splits.** We used 2017-now or 2022-now indiscriminately. Ch 6 covers purged & embargoed cross-validation.
- **Multi-asset perf.** We summarized individual-symbol performance; constructing a portfolio with weights, leverage, rebalancing, and turnover penalties comes in Ch 5 / Ch 8.
- **Universe construction.** We saw the full universe has 850 symbols with hugely varying history lengths and listing dates. Ch 4 will introduce the standard pattern: at each rebalance, re-filter by min-history, liquidity, and quote currency; gracefully handle days where the eligible set is small.